# 02 - SGLang 进阶：批量推理与性能度量

## 学习目标
- 深入理解 `sgl.function` 的批量推理机制
- 掌握多轮对话的构建方式
- 学会提取和分析推理元数据 (meta_info)
- 手动计算吞吐量、延迟等性能指标
- 编写一个完整的批量 benchmark 脚本

## 1. sgl.function 批量推理原理

### run_batch 的工作流程

```
arguments = [{"conv_messages": msgs_1}, {"conv_messages": msgs_2}, ...]
         │
         ▼
  ┌─────────────────────────────┐
  │   run_batch(arguments,       │
  │     temperature=0,           │  ← 全局参数
  │     max_new_tokens=8192,     │
  │     num_threads=16,          │  ← 并发线程数
  │     progress_bar=True)       │
  └─────────────────────────────┘
         │
         ▼
  ThreadPoolExecutor(num_threads)
    ├── thread 1 → 发送 request 1 → 等待 response 1
    ├── thread 2 → 发送 request 2 → 等待 response 2
    ├── ...
    └── thread N → 发送 request N → 等待 response N
         │
         ▼
  SGLang Server (连续批处理)
    → RadixAttention (prefix 自动缓存)
    → Batch Decode (CUDA Graph 加速)
    → (可选) EAGLE Speculative Decode
```

### 关键点
- `num_threads` 控制客户端并发数，不是 server 端 batch size
- server 端自动做 continuous batching，无需客户端干预
- 每个 request 独立返回 meta_info（含 token 统计）

In [2]:
import subprocess

MODEL_PATH = "/mnt/cfs_bj_mt/models/opensource_checkpoints/Qwen3-0.6B/"
HOST = "127.0.0.1"
PORT = 30000
TP_SIZE = 1  # 根据 GPU 数量调整

# 构建启动命令
cmd = [
    # "CUDA_VISIBLE_DEVICES=0",
    "python3", "-m", "sglang.launch_server",
    "--model-path", MODEL_PATH,
    "--host", HOST,
    "--port", str(PORT),
    "--tp-size", str(TP_SIZE),
    "--trust-remote-code",
    "--mem-fraction-static", "0.8",
]

print("Launch command:")
print(" ".join(cmd))

server_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"Server started with PID: {server_process.pid}")
print("Waiting for server to be ready...")

from sglang.utils import wait_for_server
wait_for_server(f"http://{HOST}:{PORT}")

Launch command:
python3 -m sglang.launch_server --model-path /mnt/cfs_bj_mt/models/opensource_checkpoints/Qwen3-0.6B/ --host 127.0.0.1 --port 30000 --tp-size 1 --trust-remote-code --mem-fraction-static 0.8
Server started with PID: 80014
Waiting for server to be ready...


                    NOTE: Typically, the server runs in a separate terminal.
                    In this notebook, we run the server and notebook code together, so their outputs are combined.
                    To improve clarity, the server logs are displayed in the original black color, while the notebook outputs are highlighted in blue.
                    To reduce the log length, we set the log level to warning for the server, the default log level is info.
                    We are running those notebooks in a CI environment, so the throughput is not representative of the actual performance.
                    


In [ ]:
import json
import time
import sglang as sgl
from sglang.test.test_utils import select_sglang_backend, terminate_process

# ============================================================
# 配置区
# ============================================================
HOST = "127.0.0.1"
PORT = 30000

# 连接到已运行的 SGLang server
class SimpleArgs:
    def __init__(self, host, port):
        self.backend = "srt"
        self.host = f"http://{host}"
        self.port = port
        self.model_path = None
        self.tokenizer_path = None
        self.base_url = None

args = SimpleArgs(HOST, PORT)
backend = select_sglang_backend(args)
sgl.set_default_backend(backend)
print(f"Connected to SGLang at {HOST}:{PORT}")

Connected to SGLang at 127.0.0.1:30000


## 2. 多轮对话的构建

在 `qf_mtp_eval` 项目中，benchmark 数据可能包含多轮对话。SGLang SDK 通过 role 标记来构建对话历史。

In [ ]:
@sgl.function
def multi_turn_chat(s, conv_messages):
    """Handle full conversation with system/user/assistant turns.
    
    This is the core pattern used in qf_mtp_eval's bench_sglang_eagle.py.
    The last message should be from user, and we generate the assistant reply.
    """
    for msg in conv_messages:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            s += sgl.system(content)
        elif role == "user":
            s += sgl.user(content)
        elif role == "assistant":
            s += sgl.assistant(content)
    # Generate the final assistant response
    s += sgl.assistant(sgl.gen("answer"))


# 示例：单轮对话
single_turn = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is Python?"},
]

# 示例：多轮对话
multi_turn = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is EAGLE?"},
    {"role": "assistant", "content": "EAGLE is a speculative decoding algorithm."},
    {"role": "user", "content": "How does it improve inference speed?"},
]

# 运行单轮
ret = multi_turn_chat.run(conv_messages=single_turn, temperature=0, max_new_tokens=128)
print(f"[Single-turn] Answer: {ret['answer'][:100]}")

# 运行多轮
ret = multi_turn_chat.run(conv_messages=multi_turn, temperature=0, max_new_tokens=128)
print(f"[Multi-turn]  Answer: {ret['answer'][:100]}")

## 3. get_meta_info() — 推理元数据

每个 `sgl.gen("name")` 都可以通过 `ret.get_meta_info("name")` 获取元数据。

### 返回字段说明

```python
{
    'id': '3bb9c5ead109488d...',       # 请求 ID
    'finish_reason': {'type': 'stop'},  # 结束原因
    'prompt_tokens': 37,                # prompt token 数
    'completion_tokens': 256,           # 生成的 token 数
    'cached_tokens': 0,                 # KV Cache 命中的 token 数
    'spec_verify_ct': 101,              # 推测解码验证次数 (EAGLE 特有!)
}
```

**关键字段**：
- `spec_verify_ct` — 只有开启推测解码时才有此字段，是计算 Acceptance Length 的关键

In [ ]:
# 查看 meta_info 的完整内容
ret = multi_turn_chat.run(
    conv_messages=[
        {"role": "user", "content": "Write a Python function to compute factorial."},
    ],
    temperature=0,
    max_new_tokens=256,
)

meta = ret.get_meta_info("answer")
print("=== Meta Info ===")
print(json.dumps(meta, indent=2))

print(f"\n--- Summary ---")
print(f"Prompt tokens:     {meta['prompt_tokens']}")
print(f"Completion tokens: {meta['completion_tokens']}")
print(f"Cached tokens:     {meta.get('cached_tokens', 0)}")

# 检查是否有推测解码信息
if 'spec_verify_ct' in meta:
    verify_ct = meta['spec_verify_ct']
    accept_length = meta['completion_tokens'] / verify_ct if verify_ct > 0 else 0
    print(f"Spec verify count: {verify_ct}")
    print(f"Accept length:     {accept_length:.2f}")
else:
    print("(No speculative decoding - spec_verify_ct not present)")

## 4. 性能指标计算

在 MTP 评测中，我们关注以下核心指标：

| 指标 | 公式 | 含义 |
|------|------|------|
| **Throughput** | `sum(completion_tokens) / total_latency` | 系统每秒生成的 token 数 |
| **Latency** | `end_time - start_time` | 处理全部请求的总时间 |
| **Accept Length** | `completion_tokens / spec_verify_ct` | 每次验证平均接受的 token 数 |
| **Macro AL** | `mean(per_sample_AL)` | 对各样本 AL 取平均 |
| **Micro AL** | `sum(all_tokens) / sum(all_verify_ct)` | 全局 token/验证次数比 |

In [ ]:
# 加载示例数据
import os

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
QUESTION_FILE = os.path.join(DATA_DIR, "sample_questions.jsonl")

# 读取 questions
questions = []
with open(QUESTION_FILE, "r") as f:
    for line in f:
        obj = json.loads(line)
        questions.append(obj)

print(f"Loaded {len(questions)} questions")
print(f"Example: {json.dumps(questions[0], ensure_ascii=False)[:100]}...")

In [ ]:
# 构建 arguments
arguments = []
for q in questions:
    messages = q.get("messages", [])
    if messages:
        arguments.append({"conv_messages": messages})

print(f"Prepared {len(arguments)} arguments for batch run")

# 批量推理
tic = time.perf_counter()
rets = multi_turn_chat.run_batch(
    arguments,
    temperature=0,
    max_new_tokens=512,
    num_threads=8,
    progress_bar=True,
)
latency = time.perf_counter() - tic
print(f"\nBatch completed in {latency:.2f}s")

In [ ]:
# 计算性能指标
total_completion_tokens = 0
total_prompt_tokens = 0
sample_accept_lengths = []
total_verify_ct = 0
has_speculative = False

for ret in rets:
    meta = ret.get_meta_info("answer")
    comp_tokens = meta["completion_tokens"]
    prompt_tokens = meta.get("prompt_tokens", 0)
    
    total_completion_tokens += comp_tokens
    total_prompt_tokens += prompt_tokens
    
    if "spec_verify_ct" in meta:
        has_speculative = True
        verify_ct = meta["spec_verify_ct"]
        total_verify_ct += verify_ct
        if verify_ct > 0:
            sample_al = comp_tokens / verify_ct
            sample_accept_lengths.append(sample_al)

# Throughput
throughput = total_completion_tokens / latency

print("=" * 60)
print("Performance Metrics")
print("=" * 60)
print(f"Num requests:          {len(rets)}")
print(f"Total prompt tokens:   {total_prompt_tokens}")
print(f"Total output tokens:   {total_completion_tokens}")
print(f"Latency:               {latency:.2f}s")
print(f"Throughput:            {throughput:.2f} tokens/s")

if has_speculative:
    macro_al = sum(sample_accept_lengths) / len(sample_accept_lengths) if sample_accept_lengths else 0
    micro_al = total_completion_tokens / total_verify_ct if total_verify_ct > 0 else 0
    print(f"\n--- Speculative Decoding Metrics ---")
    print(f"Macro Accept Length:   {macro_al:.3f}")
    print(f"Micro Accept Length:   {micro_al:.3f}")
    print(f"Total verify count:    {total_verify_ct}")
else:
    print(f"\n(No speculative decoding detected)")
print("=" * 60)

## 5. 完整 Benchmark 脚本示例

下面把上述知识整合为一个完整的 benchmark 函数，模仿 `qf_mtp_eval` 的 `bench_sglang_eagle.py` 核心逻辑。

In [ ]:
import uuid


def load_questions(filename, num_questions=None):
    """Load questions from JSONL file.
    
    Supports multiple formats:
    1. {"messages": [...]} — standard format
    2. {"request": {"body": {"messages": [...]}}} — nested format
    3. {"instruction": "...", "history": [...]} — legacy format
    """
    questions = []
    with open(filename, "r") as f:
        for i, line in enumerate(f):
            if num_questions and i >= num_questions:
                break
            obj = json.loads(line)
            
            messages = None
            if "messages" in obj:
                messages = obj["messages"]
            elif "request" in obj and "body" in obj["request"]:
                messages = obj["request"]["body"].get("messages", [])
            elif "instruction" in obj:
                messages = [{"role": "user", "content": obj["instruction"]}]
            
            if messages:
                questions.append({"id": i, "messages": messages})
    return questions


def run_benchmark(question_file, num_questions=None, max_new_tokens=512, num_threads=8):
    """Run a simple benchmark and return metrics."""
    # Load data
    questions = load_questions(question_file, num_questions)
    print(f"Loaded {len(questions)} questions")
    
    # Build arguments
    arguments = [{"conv_messages": q["messages"]} for q in questions]
    
    # Run batch inference
    tic = time.perf_counter()
    rets = multi_turn_chat.run_batch(
        arguments,
        temperature=0,
        max_new_tokens=max_new_tokens,
        num_threads=num_threads,
        progress_bar=True,
    )
    latency = time.perf_counter() - tic
    
    # Collect metrics
    total_output_tokens = sum(r.get_meta_info("answer")["completion_tokens"] for r in rets)
    throughput = total_output_tokens / latency
    
    has_verify = "spec_verify_ct" in rets[0].get_meta_info("answer")
    
    sample_als = []
    if has_verify:
        for r in rets:
            meta = r.get_meta_info("answer")
            vct = meta.get("spec_verify_ct", 0)
            if vct > 0:
                sample_als.append(meta["completion_tokens"] / vct)
    
    accept_length = sum(sample_als) / len(sample_als) if sample_als else 1.0
    
    result = {
        "num_requests": len(questions),
        "latency": round(latency, 3),
        "throughput": round(throughput, 3),
        "total_output_tokens": total_output_tokens,
        "accept_length": round(accept_length, 3),
        "has_speculative": has_verify,
    }
    
    return result, rets


# 运行 benchmark
result, rets = run_benchmark(QUESTION_FILE, num_questions=10, max_new_tokens=256)

print("\n=== Benchmark Result ===")
print(json.dumps(result, indent=2))

In [ ]:
# 将结果保存为 qf_mtp_eval 兼容的格式
def save_results(result, answers, questions, answer_file, result_file):
    """Save benchmark results in qf_mtp_eval compatible format."""
    # Save answers
    with open(answer_file, "w", encoding="utf-8") as f:
        for i, ret in enumerate(answers):
            meta = ret.get_meta_info("answer")
            record = {
                "question_id": questions[i]["id"],
                "answer_id": uuid.uuid4().hex,
                "model_id": "test_model",
                "input": {"messages": questions[i]["messages"]},
                "choices": {"index": 0, "answer": ret["answer"]},
                "meta": meta,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    
    # Save result summary
    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    
    print(f"Answers saved to: {answer_file}")
    print(f"Results saved to: {result_file}")


# 保存
questions_loaded = load_questions(QUESTION_FILE, 10)
save_results(
    result, rets, questions_loaded,
    answer_file=os.path.join(DATA_DIR, "my_answers.jsonl"),
    result_file=os.path.join(DATA_DIR, "my_result.json"),
)

## 6. SGLang 实用工具函数

在 `qf_mtp_eval` 项目中，使用了 SGLang 内置的工具函数来简化参数解析和后端连接。

### add_common_sglang_args_and_parse

来自 `sglang.test.test_utils`，自动添加以下参数到 argparse:
- `--backend` (srt/openai)
- `--host`
- `--port`
- `--model-path`
- `--tokenizer-path`
- `--base-url`
- `--parallel` (并发数)
- `--result-file`

### select_sglang_backend

根据 args 创建对应的 backend 连接。

In [ ]:
# 模拟 qf_mtp_eval 中的 bench_sglang_eagle.py 用法
from sglang.test.test_utils import add_common_sglang_args_and_parse, select_sglang_backend
import argparse

# 创建 parser（模拟命令行调用）
parser = argparse.ArgumentParser()
parser.add_argument("--question-file", type=str, default="question.jsonl")
parser.add_argument("--answer-file", type=str, default=None)
parser.add_argument("--num-questions", type=int, default=2000)
parser.add_argument("--temperature", type=float, default=0)
parser.add_argument("--max-gen-length", type=int, default=8192)
parser.add_argument("--thinking", action="store_true", default=False)
parser.add_argument("--mt", type=str, default="all", choices=["single", "multi", "all"])

# add_common_sglang_args_and_parse 会添加 --backend, --host, --port 等
# 并解析命令行参数
# args = add_common_sglang_args_and_parse(parser)

# 在 notebook 中模拟:
args = parser.parse_args([
    "--question-file", QUESTION_FILE,
    "--num-questions", "5",
    "--max-gen-length", "256",
])
# 手动添加 sglang 相关字段
args.backend = "srt"
args.host = f"http://{HOST}"
args.port = PORT
args.model_path = None
args.tokenizer_path = None
args.base_url = None
args.parallel = 8
args.result_file = None

print(f"Parsed args: {vars(args)}")

## 本节小结

| 知识点 | 掌握内容 |
|--------|----------|
| run_batch | 批量推理的核心 API，配合 num_threads 控制并发 |
| 多轮对话 | `sgl.system` / `sgl.user` / `sgl.assistant` 构建对话流 |
| meta_info | `get_meta_info("name")` 提取 token 统计和推测解码信息 |
| 性能指标 | throughput, latency, accept_length 的计算方式 |
| 结果格式 | answers.jsonl + result.json 兼容 qf_mtp_eval |

---
**下一节**: 03_speculative_decoding — 推测解码 (EAGLE/MTP) 原理与实践